# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### The rule, in plain words

A page is worth reviewing if it earns meaningful search visibility (volume)
and its click-through rate is weak relative to where it ranks (a sign the
snippet/title isn't converting visibility into clicks) — the same logic
behind FlyRank's quick-win and CTR-fix flags.

The baseline rule uses two observed signals, aggregated to the page level
(one row = one page, summed across all of March — not one row per page-day):

- **march_impressions**: total observed search visibility for the page in March.
- **avg_position**: the page's traffic-weighted average ranking position in March.

**Reason code:** `HIGH_VISIBILITY_OPPORTUNITY` — the page has high observed
impressions and a rankable average position, so it is a candidate for
review and possible refresh.

**Action:** `REVIEW_REFRESH`

The rule is decision-support only. It does not claim that refreshing the
page will improve performance.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Raw rows (page-days):", len(df))
print("Columns:", len(df.columns))

# Aggregate to page level FIRST — the decision unit is "review this page",
# not "review this page on this specific day"
page_agg = (
    df.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_sum_position=("gsc_sum_position", "sum")
    )
)
page_agg["avg_position"] = page_agg["march_sum_position"] / page_agg["march_impressions"]

print("Distinct pages after aggregation:", len(page_agg))
page_agg.head()

Raw rows (page-days): 9841378
Columns: 31
Distinct pages after aggregation: 331437


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_sum_position,avg_position
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,0,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,0,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,0,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9,9.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,0,NaN


### Signal 1: march_impressions (volume — behind the quick-win flag)
Verdict: CONFIRMED

Higher impression buckets should reflect stronger observed search
visibility, which is the volume signal behind FlyRank's quick-win flag
logic from the session.

In [2]:
page_agg["impressions_bucket"] = pd.cut(
    page_agg["march_impressions"],
    bins=[-1, 0, 5, 20, 100, np.inf],
    labels=["0", "1-5", "6-20", "21-100", "101+"]
)

print(page_agg["impressions_bucket"].value_counts(sort=False))

impressions_bucket
0         154699
1-5        26166
6-20       18817
21-100     30523
101+      101232
Name: count, dtype: int64


### Signal 2: CTR vs. position (behind the CTR-fix flag)
Verdict: CONFIRMED

FlyRank's CTR-fix flag logic assumes click-through rate drops as position
gets worse. I check this directly: CTR should be highest for top positions
and decline as position worsens.

In [3]:
page_agg["ctr"] = page_agg["march_clicks"] / page_agg["march_impressions"]

page_agg["position_bucket"] = pd.cut(
    page_agg["avg_position"],
    bins=[-1, 3, 10, 20, 50, 100000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

ctr_by_position = (
    page_agg.groupby("position_bucket", observed=True)["ctr"]
    .agg(["count", "mean", "median"])
)
print(ctr_by_position)

                 count      mean  median
position_bucket                         
1-3              18860  0.011696     0.0
4-10             83288  0.004873     0.0
11-20            29922  0.003285     0.0
21-50            32240  0.002379     0.0
50+              12428  0.000846     0.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
### Rule implementation

The score is: 2 * log(1 + march_impressions) — a transparent, readable
formula with no fitted weights, computed once per page for the full month.

Pages with high visibility (impressions >= 101) and a rankable average
position (<= 20) receive the reason code HIGH_VISIBILITY_OPPORTUNITY and
the action REVIEW_REFRESH. All other pages are labeled MONITOR.

This is a baseline ranking rule, not a prediction of future performance.
It uses only March data — no future-window or label-derived inputs.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

queue = page_agg[[
    "client_hash_id", "content_hash_id",
    "march_impressions", "march_clicks", "avg_position"
]].copy()

queue["score"] = 2 * np.log1p(queue["march_impressions"])

queue["reason_code"] = np.where(
    (queue["march_impressions"] >= 101) & (queue["avg_position"] <= 20),
    "HIGH_VISIBILITY_OPPORTUNITY",
    "MONITOR"
)

queue["action"] = np.where(
    queue["reason_code"] == "HIGH_VISIBILITY_OPPORTUNITY",
    "REVIEW_REFRESH",
    "MONITOR"
)

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

print("Total distinct pages in queue:", len(queue))
print(queue.head(20))

Total distinct pages in queue: 331437
             client_hash_id           content_hash_id  march_impressions  \
0   client_e547b89c05043229  content_eadb33b5df496f4a             617124   
1   client_e547b89c05043229  content_ec2e0346994fb5a5             245276   
2   client_23a62021009f63c4  content_e8a52cf3d5988c07             244931   
3   client_e547b89c05043229  content_0e03de7680314cd5             221310   
4   client_23a62021009f63c4  content_44f34c0a90047651             212404   
5   client_62f4a7e64f5e0096  content_7172a7fad43f0998             205867   
6   client_08a6a72ff48e62c0  content_e7b5dd4dff461ad2             205045   
7   client_e547b89c05043229  content_8d7d99f109e19aa2             203497   
8   client_62f4a7e64f5e0096  content_f107e54b10b43725             195997   
9   client_23a62021009f63c4  content_36e53e9c707674fc             194579   
10  client_62f4a7e64f5e0096  content_b99ea6861864dea5             194337   
11  client_e547b89c05043229  content_4ffe18112a564

In [5]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "client_hash_id", "content_hash_id",
    "march_impressions", "avg_position",
    "score", "reason_code", "action"
]

queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Saved:", "work/outputs/baseline_action_score.csv")
print("Rows written:", len(queue))

Saved: work/outputs/baseline_action_score.csv
Rows written: 331437


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Now that the queue is aggregated at the page level, each of the top 20
rows is a distinct page. For each: the action, why it's there, a
confidence note, and what would make it wrong.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

top20["confidence_note"] = (
    "Medium confidence: strong observed visibility, "
    "but the baseline does not measure content quality or actual refresh impact."
)

top20["what_would_make_it_wrong"] = (
    "The page may not have a meaningful refresh opportunity, "
    "or the observed visibility may not translate into an actionable content change."
)

review_cols = [
    "content_hash_id", "march_impressions", "avg_position",
    "score", "reason_code", "action",
    "confidence_note", "what_would_make_it_wrong"
]

print("Distinct pages in top 20:", top20["content_hash_id"].nunique())
print(top20[review_cols].to_string(index=False))

Distinct pages in top 20: 20
         content_hash_id  march_impressions  avg_position     score                 reason_code         action                                                                                                            confidence_note                                                                                                                what_would_make_it_wrong
content_eadb33b5df496f4a             617124      2.331470 26.665654 HIGH_VISIBILITY_OPPORTUNITY REVIEW_REFRESH Medium confidence: strong observed visibility, but the baseline does not measure content quality or actual refresh impact. The page may not have a meaningful refresh opportunity, or the observed visibility may not translate into an actionable content change.
content_ec2e0346994fb5a5             245276      2.757730 24.820287 HIGH_VISIBILITY_OPPORTUNITY REVIEW_REFRESH Medium confidence: strong observed visibility, but the baseline does not measure content quality or actual refresh impac

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
### Weak picks

The weakest recommendations are reviewed to see whether the rule produces
obviously poor candidates — pages with zero impressions correctly fall
into MONITOR, confirming the rule doesn't force a recommendation where
there's no visibility to act on.

### Leakage check

The baseline uses only march_impressions, march_clicks, and avg_position —
all aggregated from March data. It does not use future-window outcomes,
labels, or product-decision flags.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak_picks = queue[queue["reason_code"] == "MONITOR"].sort_values("score", ascending=True)

print("Number of MONITOR pages:", len(weak_picks))
print(weak_picks[output_cols].head(10))

Number of MONITOR pages: 254006
                 client_hash_id           content_hash_id  march_impressions  \
206842  client_fef1a8f436438636  content_e8934052490dd702                  0   
206821  client_ba65e80a1116ae41  content_32942ae784e0008e                  0   
206852  client_a60a11451483af1c  content_aebe7a856c5528f3                  0   
206851  client_ba65e80a1116ae41  content_2f3df3e17506b49f                  0   
206850  client_ba65e80a1116ae41  content_2f45f4b1da849a0d                  0   
206849  client_ba65e80a1116ae41  content_2f4c10b3b066f6e7                  0   
206848  client_ba65e80a1116ae41  content_2f5541c312aa6aaa                  0   
206847  client_ba65e80a1116ae41  content_734b6f694ac0f622                  0   
206846  client_a60a11451483af1c  content_aedabfa29f00fe90                  0   
206845  client_f623b01661d4bfe4  content_52a66e5ff89390e8                  0   

        avg_position  score reason_code   action  
206842           NaN    0.0     MONI

In [10]:
# Real weak-pick hunt: look INSIDE the recommended list for suspicious cases —
# e.g. huge impressions but near-zero clicks (possible bot/crawler traffic
# or a reporting artifact, not real search demand)
recommended = queue[queue["reason_code"] == "HIGH_VISIBILITY_OPPORTUNITY"].copy()
recommended["ctr"] = recommended["march_clicks"] / recommended["march_impressions"]

suspicious = recommended.sort_values("ctr", ascending=True).head(5)
print("Recommended pages with the LOWEST CTR despite qualifying as high-visibility:")
print(suspicious[["content_hash_id", "march_impressions", "march_clicks", "avg_position", "ctr", "score"]])

Recommended pages with the LOWEST CTR despite qualifying as high-visibility:
                content_hash_id  march_impressions  march_clicks  \
88292  content_c676e6ccfaaa1ebc                174             0   
88286  content_ec93ac94b00c7d5c                174             0   
88287  content_1a9de5661ad12dea                174             0   
55431  content_acf2fb81d02a8656                647             0   
55432  content_8fc644f32911f136                647             0   

       avg_position  ctr      score  
88292     18.005747  0.0  10.329572  
88286      7.436782  0.0  10.329572  
88287     11.821839  0.0  10.329572  
55431      3.588872  0.0  12.947781  
55432     14.522411  0.0  12.947781  


Several recommended pages (e.g. content_c676e6ccfaaa1ebc, content_acf2fb81d02a8656)
show exactly zero clicks despite 174-647 impressions — a CTR of 0.0. These
qualify for HIGH_VISIBILITY_OPPORTUNITY purely on impression volume and a
rankable position, but the complete absence of clicks suggests the
visibility may not reflect real user interest: possibly low-intent or
irrelevant queries, a poor snippet/title match, or a reporting artifact
rather than genuine search demand. This is a concrete example of why the
baseline is a starting point for human review, not a final verdict — the
rule sees impressions and position only, and cannot see click quality or
query relevance on its own.

In [9]:
used_features = ["march_impressions", "march_clicks", "avg_position"]
print("Features used by baseline:", used_features)
print("All aggregated from March data only.")
print("No future-window, outcome, or product-decision-flag columns were used.")

Features used by baseline: ['march_impressions', 'march_clicks', 'avg_position']
All aggregated from March data only.
No future-window, outcome, or product-decision-flag columns were used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.